### Abstract
In this project, I investigated timelines for beach advisories in San Diego County from the beginning ot 2025. This is my first large ArcGIS project, and a lot of my flow focused on finding, importing, and scrubbing data, both through Pro's UI and with Python APIs. This quickly became an exercise in programming syntax, incorporating SQL filtering of fetched JSON data and manipulation of Pandas dataframes.

I ended up not doing any geoprocessing for this project, as all my data sources already had the fields I wanted (no intersections required, though I did load in a California counties layer). Instead, I used a handful of the basic features of ArcGIS to visualize data and draw conclusions.

First I created charts to look at the relationship between Tijuana River pollution and beach advisories showing that, according to the California Open Data Portal (CODP), all beach closures in the area have been a direct result of Tijuana River pollution; sampling standard violations only warrant advisories.

Last I created two layers to visualize advisories in the county; the first a heat map of advisories to see what the county's beaches have tended to look like over the last year and half; the second a timeline of advisories to potentially draw conclusions about the effects of factors like ocean current on the flow of advisories.

I also exported the timeline to Excel to more effectively chart the advisories by location in attempt to identify patterns; unfortunately I didn't see anything meaningful.

### General Notes
I wanted to keep a notebook in this pro file to document what I'm doing since I'm mostly just messing around, and will therefore forget everything about everything in due time. This started as a kind of wholistic exercise in ArcGIS knowledge/Python use cases; I've been hearing a lot about the Tijuana River water issues so this was a logical place to start.

After a few Esri, Google, and Claude queries I finally found something useful for this project. I knew data sourcing would be a pain, but cut to me inspecting the HTML of various web tools on water quality for arcgis references and I can see how this is an art in it of itself.

A few data sources are worth remembering. First is the ArcGIS Online public catalog (obviously). 

I've also learned a lot about ArcGIS Enterprise and Esri's other products that facilitate hosting outside of the public catalog; SANDAG's Regional Data Warehouse (found here: https://geo.sandag.org/portal/apps/experiencebuilder/experience/?id=fad9e9c038c84f799b5378e4cc3ed068) was a good example of authoritative data storage, though while I did load in a watershed service, I haven't REALLY used anything from their portal yet.

The third is where the money is: the California Open Data Portal (found here: https://data.ca.gov/). There's a lot in here, and the documentation, particularly the Python API examples, were incredibly helpful. There's been a lot a lot to learn about data storage and the internet here; one of those things being CKAN software and how it generates all this content from the underlying data. CSV downloads, JSON APIs. Hopefully I see more CKAN later in my journey so I can stay familiar.

There was a lot of data in these services, I decided to pull monitoring stations, sample data, and beach closures. As it turns out a lot of this data gets duplicated (for example, station names and location are stored in all three services, so I don't need the monitoring station layer to cross reference). Sample data was a few million entries, so I started filtering the API with SQL to include only data from San Diego County since 1/1/2025.

After some frustrating experiences with data manipulation, it's becoming clear that unless I get very fluent in python, it's probably going to be easier to clean my data in excel with a CSV rather than load it in and convert to a feature layer immediately and try to clean it with pro's ui.

### Questions

Interestingly, there isn't any sample data after 9/9/2025, but there are beach advisories as late as 1/22/2026. So I suppose I'll mainly be looking at drawing conclusions between January and September 2025. Now I just need to... make up some questions. A few I thought of:

1. I noticed some monitoring stations don't have any samples. There are fields for station seasonal sample frequency, how accurate are those? Are there geographical patterns in data collection standards?

2. How impactful is the Tijuana River situation in the context of all beach advisories in San Diego County?

3. Can I back-calculate the water quality standards based on beach advisories and associated sample data?

4. Can advisories be used to predict future advisories in neighboring beaches? Bonus points if I could incoporate wind direction or some kind of ocean current dynamics, but I don't want to take this project that far.

### xlsx Export

So I'm much better at Excel. Some of these questions aren't spatially relevant, so I'll see how much I can get done in the xlsx. 

After a few exports I decided to export all three layers as one workbook with three sheets. I mistakenly updated the fields display on monitoring_stations before realizing I had already copied that layer into SD_mon_stats to show just stations in San Diego. At this point I'm already smarter than that, let me assign a definition query to the original monitoring_stations layer, and delete the copy before re-exporting the data.

I know how to handle data in Excel, though one optimization I made was to switch out index matching of filtered arrays with maxifs of the raw array for performance.



### Answers

1. After a brief view of the monitoring station data, plenty of the sampling frequency data lists "missing" or "not sampled". I can already see the sampling vs. monitoring station data spatially on the project map, so I don't see what would be gained by analyzing actual sampling vs sampling frequency. I do see on the map that stations missing sampling tend to be more inland (San Diego Bay, Mission Bay), as well as less visited beaches (designated park areas vs not).

2. Funny enough, I charted the counts of advisories by type and cause and Tijuana River associated contamination always and only caused closures; bacterial standards violations always and only caused postings.

   I also charted the sum of advisory duration by beach and sure enough, the two Imperial Beach locations dominated (~1,000 days of closure vs ~350 days for Silver Strand and Coronado combined).

   Finally, I updated the symbology for the beach advisories to a dynamic heat map, weighted by advisory duration. This is a clear spatial representation of the extent that the Tijuana River contributes to advisories in San Diego County as a whole.

   
3. After cleaning the data of advisories due to sampling before 01/01/2025, there remain a handful of advisories with low contamination results. A few of these advisories are associated with Tijuana River contamination, which I'm assuming is related to known runoff events, AVO monitoring, etc.  However, some advisories show low contamination in conjunction with a "Bacterial Standards Violation" advisory cause. This could be for the same reasons as above, but more investigation is probably warranted.

    Outside of small outlier values, advisories begin to appear when Enterococcus levels exceed 322 copies/100 mL (the lowest non-outlier value) for "Bacterial Standards Violation" cause (this data is better suited for drawing quantitative conclusions than Tijuana River associated advisories). I looked up what it should be and...it's 1,413 copies/100 mL according to CDPH. Well, could be worse haha. 16 of the 19 advisories testing under that threshold occurred in Silver Strand, so I wonder if something is going on there.

   One more puzzle piece to note is that monitoring stations do test for fecal and total coliforms, but after a quick internet search I decided not to use these values, as Enterococcus is widely regarded as a better measure of contamination in saltwater due to its longevity.

4. This is best investigated with sampling data over advisory data. I would have done this in pro, but I already made the mistake of importing data directly to my map and would rather do it in Excel over writing all the arcpy to create new fields to convert all the data to numeric values.

   I decided to group beaches to make this analysis simpler since this is just an exercise. A more advanced version of this analysis, if this one yields meaningful results, could include fluid dynamics and specific lat/longs of monitoring stations. Groups included Imperial Beaches, Silver Strand Beach, Coronado Beaches, and San Diego Bay.

   I then created a PivotChart following Enterococcus levels over time in the four beach groupings, which I ended up removing San Diego Bay from because of its relative insignificance. The result: it does look like there is some correlation between Enterococcus levels at the various beaches. However, it's not a very tight relationship, and on a quick glance, it doesn't look like there's any sort of time delay between elevated contamination levels at the different beaches.

### Revisit Remarks
As I'm revisiting this map to exercise publishing layers to ArcGIS online, I want to make note of how my heat map is rendering. I'm calculating an advisory duration, which for advisories without stop times, is 0, not infinity. I think it's fine excluding these from the analysis because the data looks incomplete; there's no way the advisory at silver strand that (allegedly) started on 1/1/26 is still going on. However, these are still reflected in the timeline, which I had set to run through the month of January in 2026.